In [48]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer


In [35]:
df = pd.read_parquet('/content/tasks-2026/task1/chebi_dataset_train.parquet')

In [36]:
df.head

,mol_id,SMILES,class_0,class_1,class_2,class_3,class_4,class_5,class_6,class_7,...,class_490,class_491,class_492,class_493,class_494,class_495,class_496,class_497,class_498,class_499
0,mol_12500,CCCCC/C=C\CCCCCCCC(=O)O,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,mol_15962,Cc1cc2cc(O)cc(O)c2c(C)n1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
2,mol_42147,C[C@H](CCC[C@@H](C)C=O)[C@H]1CC[C@H]2[C@@H]3CC...,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,mol_43459,CCN=C1C=CC(=C(c2ccc(NCC)cc2)c2ccc(NCC)c(C)c2)C=C1,1,1,1,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,mol_12734,C[C@H](CCC(O)=N[C@@H](Cc1c[nH]c2ccccc12)C(=O)O...,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [37]:
tqdm.pandas(desc="Generowanie deskryptorów")
names = [d[0] for d in Descriptors._descList]
calc = MoleculeDescriptors.MolecularDescriptorCalculator(names)

def extract_smiles_features(smiles_str):
    mol = Chem.MolFromSmiles(smiles_str)
    if mol:
        return list(calc.CalcDescriptors(mol))
    else:
        # Return Nones if SMILES is invalid
        return [None] * len(names)

# 2. Apply to your dataframe
# This creates a list of lists containing the 200+ features
features = df["SMILES"].progress_apply(extract_smiles_features).tolist()

# 3. Create a new dataframe from these features
df_features = pd.DataFrame(features, columns=names)

# 4. Combine with your original mol_id and classes
# We drop the original SMILES to keep it clean
df_final = pd.concat([df.drop(columns=['SMILES']), df_features], axis=1)

print(f"New shape: {df_final.shape}")

Generowanie deskryptorów:   0%|          | 0/33668 [00:00<?, ?it/s]

[13:30:32] WARNING: not removing hydrogen atom without neighbors
[13:30:32] WARNING: not removing hydrogen atom without neighbors
[13:30:32] WARNING: not removing hydrogen atom without neighbors
[13:30:32] WARNING: not removing hydrogen atom without neighbors
[13:30:32] WARNING: not removing hydrogen atom without neighbors
[13:30:32] WARNING: not removing hydrogen atom without neighbors
[13:30:41] WARNING: not removing hydrogen atom without neighbors
[13:30:41] WARNING: not removing hydrogen atom without neighbors
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/rdkit/ML/Descriptors/MoleculeDescriptors.py", line 88, in CalcDescriptors
    res[i] = fn(mol)
             ^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/rdkit/Chem/SpacialScore.py", line 72, in SPS
    return _SpacialScore(mol, normalize=normalize).score
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/rdkit/Chem/SpacialScore.py", lin

New shape: (33668, 718)


In [38]:
df_final

,mol_id,class_0,class_1,class_2,class_3,class_4,class_5,class_6,class_7,class_8,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,mol_12500,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,8,0
1,mol_15962,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
2,mol_42147,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,mol_43459,1,1,1,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,mol_12734,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33663,mol_26342,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
33664,mol_5233,1,1,1,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,1
33665,mol_36641,1,1,1,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
33666,mol_9633,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [39]:
target_cols = [f'class_{i}' for i in range(500)]

Y = df_final[target_cols].values

X_df = df_final.drop(columns=['mol_id'] + target_cols)
X = X_df.values

print(f"Kształt macierzy X (cechy): {X.shape}")
print(f"Kształt macierzy Y (etykiety): {Y.shape}")

Kształt macierzy X (cechy): (33668, 217)
Kształt macierzy Y (etykiety): (33668, 500)


In [50]:
X_df = X_df.replace([np.inf, -np.inf], np.nan)

# Zastępujemy braki danych (NaN) medianą z danej kolumny
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X_df), columns=X_df.columns)

In [51]:
threshold = 0.01 # Próg wariancji
selector = VarianceThreshold(threshold=threshold)

selector.fit(X_imputed)
X_var_filtered = X_imputed.loc[:, selector.get_support()]

print(f"Kształt po usunięciu stałych cech: {X_var_filtered.shape}")

Kształt po usunięciu stałych cech: (33668, 186)


/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1887: RuntimeWarning: overflow encountered in multiply
  sqr = np.multiply(arr, arr, out=arr, where=where)


In [52]:
corr_matrix = X_var_filtered.corr().abs()

# Wybieramy górny trójkąt macierzy korelacji
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Znajdujemy kolumny, które są skorelowane w ponad 95% z innymi
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > 0.95)]

X_uncorrelated = X_var_filtered.drop(columns=to_drop)
print(f"Kształt po usunięciu skorelowanych cech: {X_uncorrelated.shape}")
print(f"Usunięto m.in.: {to_drop[:5]}...")

Kształt po usunięciu skorelowanych cech: (33668, 149)
Usunięto m.in.: ['MaxEStateIndex', 'HeavyAtomMolWt', 'ExactMolWt', 'NumValenceElectrons', 'Chi0']...


In [53]:
scaler = RobustScaler()
X_final_scaled = scaler.fit_transform(X_uncorrelated)

In [57]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

In [83]:
num_positives = Y.sum(axis=0)
num_negatives = Y.shape[0] - num_positives

# Zabezpieczenie przed dzieleniem przez zero (dodajemy małą wartość epsilon)
epsilon = 1e-5
class_weights = num_negatives / (num_positives + epsilon)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

# Definicja prostego Datasetu w PyTorch
class MoleculeDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

In [84]:
class_weights_tensor = torch.clamp(class_weights_tensor, max=50.0)

In [91]:
class ChEMBLNet(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(ChEMBLNet, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = torch.clamp(x, min=-20.0, max=20.0)
        return self.network(x)

In [92]:
EPOCHS = 20
BATCH_SIZE = 64
LEARNING_RATE = 0.001
INPUT_DIM = X_final_scaled.shape[1]
NUM_CLASSES = Y.shape[1]

# Inicjalizacja podziału
mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(mskf.split(X_final_scaled, Y)):
    print(f"--- FOLD {fold+1} ---")

    # Tworzenie DataLoaderów
    train_dataset = MoleculeDataset(X_final_scaled[train_idx], Y[train_idx])
    val_dataset = MoleculeDataset(X_final_scaled[val_idx], Y[val_idx])

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # Inicjalizacja modelu, funkcji straty i optymalizatora
    model = ChEMBLNet(INPUT_DIM, NUM_CLASSES)

    # TUTAJ JEST KLUCZ: Przekazujemy wagi karzące za rzadkie klasy
    #criterion = nn.BCEWithLogitsLoss()
    criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights_tensor)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # Pętla epok
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0

        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            # DETEKTYW 1: Sprawdzenie danych wejściowych
            if torch.isnan(batch_x).any():
                print("🚨 ALARM: Dane wejściowe (X) zawierają NaN!")
                break
            if torch.isnan(batch_y).any():
                print("🚨 ALARM: Etykiety (Y) zawierają NaN!")
                break

            outputs = model(batch_x)

            # DETEKTYW 2: Sprawdzenie samej sieci (wagi, warstwy liniowe, BatchNorm)
            if torch.isnan(outputs).any():
                print("🚨 ALARM: Sieć wygenerowała NaN! (Problem z wagami lub eksplozja na wejściu)")
                break

            loss = criterion(outputs, batch_y)

            # DETEKTYW 3: Sprawdzenie funkcji straty
            if torch.isnan(loss).any():
                print("🚨 ALARM: Funkcja straty wygenerowała NaN! (Problem z targetami lub pos_weight)")
                break
            #outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()

        # Walidacja (obliczanie Macro F1)
        model.eval()
        val_preds = []
        val_trues = []

        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                outputs = model(batch_x)
                # Zamiana logitów na prawdopodobieństwa za pomocą Sigmoid
                probs = torch.sigmoid(outputs)
                # Tresholding na 0.5 do uzyskania etykiet 0/1
                preds = (probs > 0.5).int()

                val_preds.extend(preds.numpy())
                val_trues.extend(batch_y.numpy())

        # Obliczenie Macro F1
        macro_f1 = f1_score(val_trues, val_preds, average='macro', zero_division=0)
        print(f"Epoka {epoch+1}/{EPOCHS} | Strata: {train_loss/len(train_loader):.4f} | Val Macro F1: {macro_f1:.4f}")

    fold_results.append(macro_f1)

print(f"\nŚrednie Macro F1 z 5-krotnej walidacji: {np.mean(fold_results):.4f}")

--- FOLD 1 ---
Epoka 1/20 | Strata: 0.4091 | Val Macro F1: 0.3931
Epoka 2/20 | Strata: 0.2409 | Val Macro F1: 0.4372
Epoka 3/20 | Strata: 0.2081 | Val Macro F1: 0.4675
Epoka 4/20 | Strata: 0.1890 | Val Macro F1: 0.4741
Epoka 5/20 | Strata: 0.1762 | Val Macro F1: 0.5002
Epoka 6/20 | Strata: 0.1676 | Val Macro F1: 0.5075
Epoka 7/20 | Strata: 0.1594 | Val Macro F1: 0.5060
Epoka 8/20 | Strata: 0.1542 | Val Macro F1: 0.5213
Epoka 9/20 | Strata: 0.1492 | Val Macro F1: 0.5374
Epoka 10/20 | Strata: 0.1436 | Val Macro F1: 0.5336
Epoka 11/20 | Strata: 0.1407 | Val Macro F1: 0.5516
Epoka 12/20 | Strata: 0.1367 | Val Macro F1: 0.5532
Epoka 13/20 | Strata: 0.1345 | Val Macro F1: 0.5594
Epoka 14/20 | Strata: 0.1307 | Val Macro F1: 0.5551
Epoka 15/20 | Strata: 0.1290 | Val Macro F1: 0.5706
Epoka 16/20 | Strata: 0.1266 | Val Macro F1: 0.5635
Epoka 17/20 | Strata: 0.1251 | Val Macro F1: 0.5651
Epoka 18/20 | Strata: 0.1225 | Val Macro F1: 0.5734
Epoka 19/20 | Strata: 0.1203 | Val Macro F1: 0.5784
Epoka 

In [94]:
df_test = pd.read_parquet('/content/tasks-2026/task1/chebi_dataset_test_empty.parquet')

In [95]:
df_test['SMILES'].progress_apply(lambda x: x) # Inicjalizacja tqdm pandas
features_test = df_test["SMILES"].progress_apply(extract_smiles_features).tolist()
df_features_test = pd.DataFrame(features_test, columns=names)

# Przygotowanie ramki wejściowej (tylko cechy chemiczne)
X_test_df = df_features_test.copy()

Generowanie deskryptorów:   0%|          | 0/11223 [00:00<?, ?it/s]

Generowanie deskryptorów:   0%|          | 0/11223 [00:00<?, ?it/s]

[15:11:23] WARNING: not removing hydrogen atom without neighbors
[15:11:23] WARNING: not removing hydrogen atom without neighbors
[15:11:43] WARNING: not removing hydrogen atom without neighbors
[15:11:43] WARNING: not removing hydrogen atom without neighbors
[15:11:43] WARNING: not removing hydrogen atom without neighbors
[15:11:43] WARNING: not removing hydrogen atom without neighbors
[15:11:43] WARNING: not removing hydrogen atom without neighbors
[15:11:43] WARNING: not removing hydrogen atom without neighbors
[15:11:43] WARNING: not removing hydrogen atom without neighbors
[15:11:43] WARNING: not removing hydrogen atom without neighbors
[15:11:43] WARNING: not removing hydrogen atom without neighbors
[15:11:43] WARNING: not removing hydrogen atom without neighbors
[15:11:52] WARNING: not removing hydrogen atom without neighbors
[15:11:52] WARNING: not removing hydrogen atom without neighbors
[15:11:52] WARNING: not removing hydrogen atom without neighbors
[15:11:52] WARNING: not r

In [96]:
X_test_df = X_test_df.replace([np.inf, -np.inf], np.nan)

# 2. Imputacja braków danych (używa mediany ze zbioru TRENINGOWEGO)
X_test_imputed = pd.DataFrame(imputer.transform(X_test_df), columns=X_test_df.columns)

# 3. Usuwanie cech o stałej wariancji (usuwa te same kolumny, co w treningu)
X_test_var_filtered = X_test_imputed.loc[:, selector.get_support()]

# 4. Usuwanie cech skorelowanych (używamy tej samej listy 'to_drop' co wcześniej)
X_test_uncorrelated = X_test_var_filtered.drop(columns=to_drop)

# 5. Skalowanie (używa rozstępów zdefiniowanych na zbiorze TRENINGOWYM)
X_test_final_scaled = scaler.transform(X_test_uncorrelated)

# 6. Czyszczenie ukrytych NaN (zabezpieczenie przed błędem z poprzedniej rozmowy)
X_test_final_scaled = np.nan_to_num(X_test_final_scaled, nan=0.0, posinf=0.0, neginf=0.0)

print(f"Kształt danych testowych gotowych do modelu: {X_test_final_scaled.shape}")

Kształt danych testowych gotowych do modelu: (11223, 149)


In [97]:
full_train_dataset = MoleculeDataset(X_final_scaled, Y)
full_train_loader = DataLoader(full_train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

# Inicjalizacja modelu ostatecznego
final_model = ChEMBLNet(INPUT_DIM, NUM_CLASSES)
criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights_tensor)
optimizer = optim.Adam(final_model.parameters(), lr=LEARNING_RATE)

print("Rozpoczynam trening modelu ostatecznego na pełnych danych...")
for epoch in range(EPOCHS):
    final_model.train()
    train_loss = 0.0

    for batch_x, batch_y in full_train_loader:
        optimizer.zero_grad()
        outputs = final_model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(final_model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()

    print(f"Epoka {epoch+1}/{EPOCHS} ukończona. Strata: {train_loss/len(full_train_loader):.4f}")

# ---------------------------------------------------------
# PREDYKCJA NA ZBIORZE TESTOWYM
# ---------------------------------------------------------

# Przekonwertowanie przetworzonych danych testowych na tensor PyTorch
X_test_tensor = torch.tensor(X_test_final_scaled, dtype=torch.float32)

# Ustawienie modelu w tryb predykcji
final_model.eval()
test_preds = []

print("Generowanie predykcji dla pliku testowego...")
with torch.no_grad():
    # Przetwarzamy dane w batchach, żeby nie zapchać pamięci RAM
    for i in range(0, len(X_test_tensor), BATCH_SIZE):
        batch_x = X_test_tensor[i:i+BATCH_SIZE]
        outputs = final_model(batch_x)
        probs = torch.sigmoid(outputs)

        # Binarizacja z progiem 0.5 (zgodnie z wymogiem predykcji binarnych)
        preds = (probs > 0.5).int()
        test_preds.extend(preds.numpy())

# Konwersja z powrotem do formy DataFrame
test_preds_df = pd.DataFrame(test_preds, columns=target_cols)

# Doklejenie oryginalnego ID cząsteczki
final_submission = pd.concat([df_test[['mol_id']].reset_index(drop=True), test_preds_df], axis=1)

# Zmiana: Zapis do pliku PARQUET, co jest zgodne z plikiem referencyjnym hackathonu
nazwa_pliku_wyjsciowego = 'submission_chebi.parquet'
final_submission.to_parquet(nazwa_pliku_wyjsciowego, index=False)

print(f"Plik z predykcjami '{nazwa_pliku_wyjsciowego}' został pomyślnie zapisany i jest gotowy do wysłania!")

Rozpoczynam trening modelu ostatecznego na pełnych danych...
Epoka 1/20 ukończona. Strata: 0.3776
Epoka 2/20 ukończona. Strata: 0.2270
Epoka 3/20 ukończona. Strata: 0.1964
Epoka 4/20 ukończona. Strata: 0.1807
Epoka 5/20 ukończona. Strata: 0.1692
Epoka 6/20 ukończona. Strata: 0.1608
Epoka 7/20 ukończona. Strata: 0.1542
Epoka 8/20 ukończona. Strata: 0.1487
Epoka 9/20 ukończona. Strata: 0.1440
Epoka 10/20 ukończona. Strata: 0.1405
Epoka 11/20 ukończona. Strata: 0.1361
Epoka 12/20 ukończona. Strata: 0.1336
Epoka 13/20 ukończona. Strata: 0.1304
Epoka 14/20 ukończona. Strata: 0.1287
Epoka 15/20 ukończona. Strata: 0.1255
Epoka 16/20 ukończona. Strata: 0.1227
Epoka 17/20 ukończona. Strata: 0.1209
Epoka 18/20 ukończona. Strata: 0.1202
Epoka 19/20 ukończona. Strata: 0.1179
Epoka 20/20 ukończona. Strata: 0.1156
Generowanie predykcji dla pliku testowego...
Plik z predykcjami 'submission_chebi.parquet' został pomyślnie zapisany i jest gotowy do wysłania!


In [108]:
final_submission = pd.concat([df_test[['mol_id', 'SMILES']].reset_index(drop=True), test_preds_df], axis=1)

In [111]:
nazwa_pliku_wyjsciowego = 'cheb_test_submission_empty.parquet'
final_submission.to_parquet(nazwa_pliku_wyjsciowego, index=False)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')